# 04 · Diffusion Models — DDPM from Scratch

> **Source notes:** `DiffusionModels.md`

Build a working DDPM that generates handwritten-digit-style images from pure noise.

This notebook builds every component:
- **Forward process** — visualise how noise accumulates step by step
- **Noise schedule** — linear and cosine schedules
- **U-Net denoiser** — minimal architecture that runs on CPU
- **Training loop** — MSE loss on noise prediction
- **Sampling loop** — DDPM reverse process from noise → image
- **PixelSmith v3** — generate new images from pure noise

**All local. No GPU needed.** Training runtime: ~3–5 minutes on CPU.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch", "torchvision", "matplotlib", "numpy", "-q"], check=True)
print("Packages ready.")

## 1 · The Forward Process — Adding Noise

The forward process $q(x_t | x_0) = \mathcal{N}(\sqrt{\bar{\alpha}_t}\, x_0,\, (1-\bar{\alpha}_t)\mathbf{I})$ lets us jump to any noise level in one operation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# ── Noise schedule ────────────────────────────────────────────────────────────
T_STEPS = 1000

# Linear schedule (DDPM paper defaults)
beta_start, beta_end = 1e-4, 0.02
betas      = torch.linspace(beta_start, beta_end, T_STEPS)   # (T,)
alphas     = 1.0 - betas                                       # (T,)
alpha_bar  = torch.cumprod(alphas, dim=0)                      # (T,) — cumulative product
sqrt_ab    = alpha_bar.sqrt()                                  # √ᾱ_t
sqrt_1mab  = (1 - alpha_bar).sqrt()                            # √(1-ᾱ_t)

print("Noise schedule: linear DDPM")
print(f"β range: [{betas[0]:.6f}, {betas[-1]:.4f}]")
print(f"\nSignal-to-noise ratio at key steps:")
for t_idx in [0, 100, 250, 500, 750, 999]:
    snr = (sqrt_ab[t_idx] / sqrt_1mab[t_idx]).item()
    print(f"  t={t_idx+1:4d}: √ᾱ_t={sqrt_ab[t_idx]:.4f}  √(1-ᾱ_t)={sqrt_1mab[t_idx]:.4f}  SNR={snr:.4f}")

In [ ]:
def q_sample(x0: torch.Tensor, t: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Sample x_t given x_0 and t using the closed-form forward process."""
    # t is a (batch,) tensor of integer timestep indices
    eps     = torch.randn_like(x0)
    s_ab    = sqrt_ab[t].view(-1, 1, 1, 1)    # √ᾱ_t  broadcastable to image shape
    s_1mab  = sqrt_1mab[t].view(-1, 1, 1, 1)  # √(1-ᾱ_t)
    x_t     = s_ab * x0 + s_1mab * eps
    return x_t, eps  # return noisy image AND the noise used


# Visualise forward process on a single MNIST digit
dataset = torchvision.datasets.MNIST(
    root="/tmp/mnist", train=True, download=True,
    transform=T.Compose([T.ToTensor(), T.Lambda(lambda x: x * 2 - 1)])  # [0,1] → [-1,1]
)

x0, label = dataset[42]   # a specific digit
x0 = x0.unsqueeze(0)      # (1, 1, 28, 28)

timesteps_to_show = [0, 100, 200, 400, 600, 800, 999]
fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(14, 2.5))

for ax, t_idx in zip(axes, timesteps_to_show):
    t_tensor = torch.tensor([t_idx])
    x_t, _ = q_sample(x0, t_tensor)
    img     = x_t[0, 0].numpy()  # (28, 28)
    ax.imshow(img, cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"t={t_idx+1}\n√ᾱ={sqrt_ab[t_idx]:.3f}")
    ax.axis("off")

plt.suptitle(f'Forward noising process: digit "{label}" → noise', y=1.05)
plt.tight_layout()
plt.show()

print(f"At t=1:    digit clearly visible (√ᾱ ≈ {sqrt_ab[0]:.3f})")
print(f"At t=500:  digit faint (√ᾱ ≈ {sqrt_ab[499]:.3f})")
print(f"At t=1000: pure noise (√ᾱ ≈ {sqrt_ab[999]:.3f} ≈ 0)")

## 2 · The U-Net Denoiser

A minimal U-Net that can run on CPU. The key components:
- **Encoder** — downsamples spatial dimension, increases channels
- **Bottleneck** — processes most compressed representation
- **Decoder** — upsamples, concatenates skip connections from encoder
- **Timestep embedding** — sinusoidal embedding of $t$, added to every layer

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """Positional encoding for timestep t (same as Transformer pos encoding)."""
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        freqs = torch.exp(
            -torch.arange(half, device=t.device) * (np.log(10000) / (half - 1))
        )
        args  = t[:, None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)  # (batch, dim)


class ResBlock(nn.Module):
    """Residual block with time embedding injection."""
    def __init__(self, in_ch: int, out_ch: int, time_dim: int):
        super().__init__()
        self.conv1  = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(time_dim, out_ch)
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h  = F.silu(self.conv1(x))
        h  = h + self.t_proj(t_emb)[:, :, None, None]  # broadcast time over H,W
        h  = F.silu(self.conv2(h))
        return h + self.skip(x)


class MiniUNet(nn.Module):
    """Minimal U-Net for MNIST 28×28, single channel."""
    def __init__(self, base_ch: int = 32, time_dim: int = 64):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim * 4),
            nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim),
        )
        C = base_ch
        # Encoder
        self.enc1 = ResBlock(1,   C,   time_dim)  # 28×28
        self.enc2 = ResBlock(C,   C*2, time_dim)  # 14×14
        self.enc3 = ResBlock(C*2, C*4, time_dim)  # 7×7
        # Bottleneck
        self.bot  = ResBlock(C*4, C*4, time_dim)  # 7×7
        # Decoder
        self.dec3 = ResBlock(C*8, C*2, time_dim)  # 7→14×14  (skip from enc3)
        self.dec2 = ResBlock(C*4, C,   time_dim)  # 14→28×14  (skip from enc2)
        self.dec1 = ResBlock(C*2, C,   time_dim)  # 28×28  (skip from enc1)
        self.out  = nn.Conv2d(C, 1, 1)            # predict noise, same spatial
        self.down = nn.MaxPool2d(2)
        self.up   = nn.Upsample(scale_factor=2, mode="nearest")

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_embed(t)            # (B, time_dim)
        # Encode with skip connections
        s1 = self.enc1(x, t_emb)             # (B, C,   28, 28)
        s2 = self.enc2(self.down(s1), t_emb) # (B, C*2, 14, 14)
        s3 = self.enc3(self.down(s2), t_emb) # (B, C*4, 7,  7)
        # Bottleneck
        b  = self.bot(s3, t_emb)             # (B, C*4, 7,  7)
        # Decode
        d3 = self.dec3(torch.cat([self.up(b), s3], 1), t_emb)  # (B, C*2, 14, 14)
        d2 = self.dec2(torch.cat([self.up(d3), s2], 1), t_emb) # (B, C,   28, 28)
        d1 = self.dec1(torch.cat([d2, s1], 1), t_emb)          # (B, C,   28, 28)
        return self.out(d1)                   # (B, 1, 28, 28) — predicted noise


model = MiniUNet(base_ch=32, time_dim=64)
total = sum(p.numel() for p in model.parameters())
print(f"MiniUNet total parameters: {total:,}")

# Test forward pass
x_test = torch.randn(4, 1, 28, 28)
t_test = torch.randint(0, T_STEPS, (4,))
eps_hat = model(x_test, t_test)
print(f"Input: {x_test.shape}  →  Predicted noise: {eps_hat.shape}  (same shape)")

## 3 · Training Loop

In [ ]:
EPOCHS     = 20      # ~3-5 min on CPU
BATCH_SIZE = 256
LR         = 2e-4

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
optim  = torch.optim.Adam(model.parameters(), lr=LR)

losses = []
model.train()

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for x0_batch, _ in loader:
        # 1. Sample random timesteps
        t = torch.randint(0, T_STEPS, (x0_batch.shape[0],))
        # 2. Add noise (closed-form forward process)
        x_t, eps = q_sample(x0_batch, t)
        # 3. Predict noise with U-Net
        eps_pred = model(x_t, t)
        # 4. MSE loss on noise prediction
        loss = F.mse_loss(eps_pred, eps)
        # 5. Optimise
        optim.zero_grad()
        loss.backward()
        optim.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(loader)
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  loss={avg_loss:.4f}")

print("\nTraining complete.")

In [ ]:
# Plot training curve
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS+1), losses, "b-o", ms=4)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss (noise prediction)")
plt.title("DDPM Training Loss — noise prediction MSE")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final loss: {losses[-1]:.4f}")
print("Lower = model predicts noise more accurately = better image quality")

## 4 · Sampling — DDPM Reverse Process

Starting from pure Gaussian noise, iteratively denoise using the trained U-Net.

In [ ]:
@torch.no_grad()
def ddpm_sample(model: nn.Module, n_samples: int = 16,
                img_size: int = 28, channels: int = 1,
                collect_every: int = 100) -> tuple:
    """DDPM ancestral sampling: x_T ~ N(0,I) → x_0."""
    model.eval()
    x = torch.randn(n_samples, channels, img_size, img_size)
    history = []  # track intermediate states
    
    for t_idx in reversed(range(T_STEPS)):
        t_batch  = torch.full((n_samples,), t_idx, dtype=torch.long)
        eps_pred = model(x, t_batch)  # predict noise
        
        # Compute posterior mean μ_θ
        beta_t    = betas[t_idx]
        ab_t      = alpha_bar[t_idx]
        alpha_t   = alphas[t_idx]
        
        # Predicted x_0
        x0_pred  = (x - (1 - ab_t).sqrt() * eps_pred) / ab_t.sqrt()
        x0_pred  = x0_pred.clamp(-1, 1)
        
        # Posterior mean (DDPM sampling)
        if t_idx > 0:
            ab_prev  = alpha_bar[t_idx - 1]
            beta_tilde = beta_t * (1 - ab_prev) / (1 - ab_t)  # posterior variance
            mu = (ab_prev.sqrt() * beta_t / (1 - ab_t)) * x0_pred + \
                 (alpha_t.sqrt() * (1 - ab_prev) / (1 - ab_t)) * x
            noise = torch.randn_like(x)
            x = mu + beta_tilde.sqrt() * noise
        else:
            x = x0_pred  # final step: no noise
        
        if t_idx % collect_every == 0:
            history.append((t_idx, x.clone()))
    
    return x, history


print("Sampling 16 images via DDPM (1000 steps)...")
generated, history = ddpm_sample(model, n_samples=16, collect_every=100)
print(f"Generated images shape: {generated.shape}")
print(f"Value range: [{generated.min():.3f}, {generated.max():.3f}]")

In [ ]:
# Visualise: show 4×4 grid of generated images
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    img = generated[i, 0].numpy()
    ax.imshow(img, cmap="gray", vmin=-1, vmax=1)
    ax.axis("off")

plt.suptitle("PixelSmith v3 — DDPM generated images from pure noise", y=1.02)
plt.tight_layout()
plt.show()

# Visualise the denoising trajectory for one sample
n_hist    = len(history)
fig, axes = plt.subplots(1, n_hist, figsize=(3 * n_hist, 3))
for ax, (t_val, x_hist) in zip(axes, sorted(history, reverse=True)):
    ax.imshow(x_hist[0, 0].numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"t={t_val}")
    ax.axis("off")

plt.suptitle("Denoising trajectory: noise → recognisable digit", y=1.05)
plt.tight_layout()
plt.show()

## 5 · Summary — PixelSmith v3

```
┌──────────────────────────────────────────────────────────────────────────────┐
│ PixelSmith v3 — DDPM Generator                                                │
│                                                                                │
│  TRAINING                                                                      │
│    x_0 (real) + ε (noise) + t (step)                                          │
│        → q(x_t|x_0) = one-step forward jump                                  │
│        → U-Net(x_t, t) predicts ε                                             │
│        → Loss = MSE(ε, ε̂)                                                     │
│                                                                                │
│  INFERENCE                                                                     │
│    x_T ~ N(0, I)                                                              │
│        → 1000 × [predict noise, compute μ, sample x_{t-1}]                   │
│        → x_0  ← generated image                                              │
│                                                                                │
│  LIMITATION: unconditional generation — no control over what is generated.   │
│  Next: add text conditioning via cross-attention (Ch.5 + Ch.7)               │
└──────────────────────────────────────────────────────────────────────────────┘
```

**Next:** [GuidanceConditioning.md](../GuidanceConditioning/GuidanceConditioning.md) — learn how classifier-free guidance (CFG) adds text control to this denoising process.